<a href="https://colab.research.google.com/github/tatyanalitvin/python_for_ds_tasks/blob/dev/Unit_06/%D0%92%D0%B8%D0%BA%D0%BE%D1%80%D0%B8%D1%81%D1%82%D0%B0%D0%BD%D0%BD%D1%8F_%D0%BF%D1%80%D0%BE%D0%BC%D0%BF%D1%82%D1%96%D0%B2_%D1%96_%D0%B0%D0%B3%D0%B5%D0%BD%D1%82%D1%96%D0%B2_%D0%B2_Langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [1]:
!pip install -q langchain langchain-openai langchain-huggingface langchain-experimental
!pip install -q langchain_community duckduckgo_search
!pip install -U ddgs

In [2]:
import os

from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

from langchain_core.prompts import PromptTemplate

from langchain_community.tools import DuckDuckGoSearchRun
from langchain_experimental.tools.python.tool import PythonREPLTool

from langchain.agents import create_agent


# Tasks

## Завдання 1: Виклик LLM з базовим промптом

**Мета:** навчитися викликати LLM через LangChain зі звичайним текстовим промптом.

**Що потрібно зробити:**

1. Створіть промпт, який дозволяє отримати інформацію простою мовою на тему "Квантові обчислення". Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

2. Обмежте відповідь до 200 символів і пропишіть в промпті, аби відповідь була короткою (це зекономить вам час і гроші на згенеровані токени).

3. Встановіть своє значення температури на власний розсуд (тут немає правильного чи неправильного значення) і напишіть коментарем, чому ви обрали саме таке значення для цього завдання.

**Вибір моделі:** можна скористатись як моделлю з HuggingFace, так і ChatGPT будь-якої версії, яка вам до вподоби і пасує за прайсингом. В обох випадках потрібно імпортувати відповідний клас з LangChain для виклику LLM за API.

**Мова запитів:** промпти можна писати як українською, так і англійською — орієнтуйтесь на те, де і для чого ви хочете потім використовувати цей проєкт. У розв'язках промпти — українською.

---
**🔐 Як безпечно зберігати і підвантажувати API-ключі**




API-токен потрібно зчитувати з безпечного джерела, а **не хардкодити в ноутбуці**. Якщо хтось отримає доступ до вашого ключа, він буде витрачати токени за ваш рахунок, а вам це не треба :)

Є кілька способів. Перший ми використовували на лекції, ще два для розширення вашого розуміння, як ще це можна зробити і що шлях не лише один. Для виконання цього ДЗ можете використовувати будь-який спосіб підвантаження ключів у ноутбук.

**Спосіб 1: Файл `creds.json` (рекомендований)**

Створіть файл `creds.json` з вашими ключами, завантажте його в Google Colab під час роботи, але **не здавайте** цей файл у ДЗ і **не комітьте** в git.

```python
import json
with open("creds.json") as f:
    creds = json.load(f)
api_key = creds["HF_TOKEN"]
```

**Спосіб 2: Google Colab Secrets**

У лівій панелі Colab натисніть іконку 🔑 (Secrets) → "Add new secret" → введіть назву (наприклад, `HF_TOKEN`) та значення ключа → увімкніть тогл доступу для ноутбука.

```python
from google.colab import userdata
api_key = userdata.get("HF_TOKEN")
```

Зручно тим, що ключ зберігається в акаунті і доступний у всіх ваших ноутбуках. Мінус — при кожній новій сесії потрібно перевірити, що доступ увімкнено.

**Спосіб 3: Google AI Studio (для Gemini API)**

Якщо працюєте з моделями Google Gemini, отримати безкоштовний API-ключ можна в [Google AI Studio](https://aistudio.google.com/app/apikey): увійдіть з Google-акаунтом → натисніть "Get API key" → "Create API key". Далі використовуйте ключ через будь-який із способів вище.

---

In [3]:
from google.colab import userdata
os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get("HF_TOKEN")


In [4]:
# llm = ChatOpenAI(
#     model="gpt-4o-mini",
#     temperature=0.3,
#     max_tokens=200,
# )

endpoint_task1 = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    max_new_tokens=200,
    temperature=0.3, # less creative but no entirely 0 to let answer more natural
    repetition_penalty=1.03,
)

llm_task1 = ChatHuggingFace(llm=endpoint_task1)

In [5]:
prompt = f"""
Explain 'Quantum computing' in ONE dense sentence under 200 characters.
The sentence MUST explicitly contain:
(1) a short definition,
(2) one key advantage,
(3) one current research direction.
Output only the sentence, no yapping.
"""

In [6]:
response = llm_task1.invoke(prompt)
text = response.content[:200]
print(text)
print("---")
print(f"Answer len: {len(text)} chars")

Quantum computing is a revolutionary technology utilizing qubits to solve complex problems exponentially faster, offering unparalleled speedup in simulations, and currently, researchers are exploring 
---
Answer len: 200 chars


## Завдання 2: Створення параметризованого промпта для генерації тексту

Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [7]:
endpoint_task2 = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    max_new_tokens=200,
    temperature=0.3,
    repetition_penalty=1.03,
)

llm_task2 = ChatHuggingFace(llm=endpoint_task2)

In [8]:
prompt_template = PromptTemplate.from_template(
    f"""Explain the topic '{{topic}}' in simple terms.
In your answer, mention the definition, key advantages, and current research directions.
Be concise, up to 300 characters.
"""
)


In [9]:
# LCEL
chain = prompt_template | llm_task2

topics = [
    "Баєсівські методи в машинному навчанні",
    "Трансформери в машинному навчанні",
    "Explainable AI",
]

for topic in topics:
    result = chain.invoke({"topic": topic})
    print(f"=== {topic} ===")
    print(result.content)
    print()

=== Баєсівські методи в машинному навчанні ===
"Баєсівські методи в машинному навчанні" (Bayesian methods in machine learning) - це підхід, який використовує статистичні моделі для навчання та прогнозування. 

Визначення: Баєсівські методи використовують вірогідність для навчання моделей, поєднуючи дані спостереження та попередні знання.

Ключові переваги:

- Може обробляти нечіткі дані та розріджені дані
- Дозволяє інтегрувати попередні знання та експертні оцінки
- Можливість оцінки вірогідності різних гіпотез

Поточні напрямки досліджень: застосування Баєсівських методів в глибокому навчанні, обробці природної мови та обробці зображень.

=== Трансформери в машинному навчанні ===
"Трансформери в машинному навчанні" (Transformers in Machine Learning) - це тип нейронної мережі, що використовує механізм трансформації для обробки послідовних даних. 

Вона має такі переваги:
- ефективна обробка послідовних даних;
- можливість обробляти довгі послідовності;
- добре працює на великих даних.


## Завдання 3: Використання агента для автоматизації процесів

Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [10]:
!pip install -q langchain_community duckduckgo_search

In [11]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Obama's first name?")

'Barack Hussein Obama II[a] (born August 4, 1961) is an American politician who served as the 44th president of the United States from 2009 to 2017. A member of the Democratic Party, he was the first African American president. Jan 6, 2026 · The truth is straightforward: Barack Obama has never legally changed his name. He was born Barack Hussein Obama II on August 4, 1961, in Honolulu, Hawaii. This name has remained consistent throughout his life and career, symbolizing his familial heritage and personal identity. Barack Obama’s parents married while students at the University of Hawaii. His father, Barack Obama, Sr., a Kenyan, became an economist in the government of Kenya. His mother, S. Ann Dunham, became an anthropologist. They divorced in 1964. Ann then married (and later divorced) another foreign student, Indonesian Lolo Soetoro. Where did Barack Obama attend school? Barack Obama graduated from Punahou School, an elite academy in Honolulu, and then attended Occidental College bef

In [12]:
endpoint_task3 = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    temperature=0,
    repetition_penalty=1.05,
)
model_task3 = ChatHuggingFace(llm=endpoint_task3)

search_tool = DuckDuckGoSearchRun()

In [13]:
agent_prompt = f"""You are a research assistant. You MUST call the search tool at least once before answering — do not rely on memory. If the first search is not informative, refine the query and search again.

After gathering results, return EXACTLY 5 recent scientific publications as a numbered list. For each item include:
1) Title,
2) Authors,
3) a 1-2 sentence description grounded in the search results.

If the search tool returns fewer than 5 usable publications, list only what you actually found and say so — DO NOT fabricate titles, authors, or venues."""


agent = create_agent(
    model_task3,
    tools=[search_tool],
    system_prompt=agent_prompt,
)

In [14]:
query = f"""Find 5 of the most recent scientific publications on the topic of artificial intelligence (AI / machine learning) from 2025-2026.
For each paper give the title, authors, and a short description."""

result = agent.invoke({"messages": [{"role": "user", "content": query}]})

In [15]:
print("=== Message trace ===")
for m in result["messages"]:
    role = type(m).__name__
    content = (m.content or "")[:300]
    print(f"[{role}] {content}")
    tool_calls = getattr(m, "tool_calls", None)
    if tool_calls:
        for tc in tool_calls:
            print(f"  -> tool_call: {tc.get('name')} args={tc.get('args')}")

print("\n=== Final answer ===")
print(result["messages"][-1].content)

=== Message trace ===
[HumanMessage] Find 5 of the most recent scientific publications on the topic of artificial intelligence (AI / machine learning) from 2025-2026. 
For each paper give the title, authors, and a short description.
[AIMessage] 
  -> tool_call: duckduckgo_search args={'query': 'recent scientific publications on AI machine learning 2025-2026'}
[ToolMessage] Google publishes hundreds of research papers each year. Publishing our work enables us to collaborate and share ideas with, as well as learn from, the broader scientific…Cesar Magalhaes. Hamza Harkous. Transactions on Machine Learning Research (2026). Publications. Mercury Machine Learning Lab. 2026
[AIMessage] 
  -> tool_call: duckduckgo_search args={'query': 'research on machine-driven peer review risk to creativity 2026'}
[ToolMessage] The integration of large language models (LLMs) into peer review raises a concern beyond authorship and detection: the potential cascading automation of the entire editorial process

## Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [16]:
search_tool = DuckDuckGoSearchRun()
python_tool = PythonREPLTool()

# Llama-3.3-70B-Instruct is better ReAct з python_tool + search_tool.
endpoint_task4 = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.3-70B-Instruct",
    task="text-generation",
    max_new_tokens=1024,
    temperature=0,
)
model_task4 = ChatHuggingFace(llm=endpoint_task4)

In [17]:
agent = create_agent(
    model_task4,
    tools=[search_tool, python_tool],
    system_prompt=f"""You are a quantitative business analyst.
    You MUST follow this workflow strictly:

STEP 1: Parse the historical numbers from the user's query.
STEP 2: Call the python_tool to compute a baseline forecast. At minimum: compute year-over-year growth rates, the average growth rate, and a linear-trend projection for the next year. Print intermediate values.
STEP 3: Call the search_tool to gather up-to-date qualitative context (e.g. Brazil orange harvest / weather / frost 2025-2026, global orange demand, inflation). Do one or two focused searches.
STEP 4: Call the python_tool again to apply a reasonable adjustment factor to the baseline (explain the factor explicitly, e.g. -5% for adverse weather).
STEP 5: Return a final answer that contains:
  * a concrete numeric forecast for 2026 (in tonnes),
  * the baseline vs adjusted number,
  * the assumptions and adjustment factors used,
  * the sources from the search results.

You MUST produce a number. If, after following the steps, you genuinely cannot produce a defensible number, explain exactly which data is missing. Do not give a purely qualitative answer.""",
)

In [19]:
user_query = (
    "Ми експортуємо апельсини з Бразилії. В 2022 експортували 200т, "
    "в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. Зроби оцінку "
    "скільки ми зможемо експортувати апельсинів в 2026 враховуючи "
    "погодні умови в Бразилії і попит на апельсини в світі виходячи "
    "з економічної ситуації. Якщо не можеш дати оцінку — поясни чому."
)
result = agent.invoke(
    {"messages": [{"role": "user", "content": user_query}]}
)

print(result["messages"][-1].content)

За результатами розрахунків і враховуючи погодні умови в Бразилії та попит на апельсини у світі, в 2026 році очікується експорт апельсинів у кількості 216 тисяч тонн. Очікувана експортна кількість визначена на основі лінійної тенденції прогнозування, враховуючи середню ставку росту експорту за попередні роки. До цього прогнозу застосовано корекцію -5% через погодні умови в Бразилії, що сприяли зниженню урожайності апельсинів. Таким чином, остаточна оцінка склала 216 тисяч тонн.


In [20]:
print("=== Message trace ===")
for m in result["messages"]:
    role = type(m).__name__
    content = (m.content or "")[:300]
    print(f"[{role}] {content}")
    tool_calls = getattr(m, "tool_calls", None)
    if tool_calls:
        for tc in tool_calls:
            print(f"  -> tool_call: {tc.get('name')} args={tc.get('args')}")

print("\n=== Final answer ===")
print(result["messages"][-1].content)

=== Message trace ===
[HumanMessage] Ми експортуємо апельсини з Бразилії. В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації. Якщо не можеш дати оцінку — поя
[AIMessage] 
  -> tool_call: Python_REPL args={'query': 'import numpy as np\nyear = np.array([2022, 2023, 2024, 2025])\nexports = np.array([200, 190, 210, 220])\ngrowth_rates = np.diff(exports) / exports[:-1]\naverage_growth_rate = np.mean(growth_rates)\nlinear_trend_projection = exports[-1] + average_growth_rate * exports[-1]\nprint("Year-over-year growth rates: ", growth_rates)\nprint("Average growth rate: ", average_growth_rate)\nprint("Linear-trend projection for 2026: ", linear_trend_projection)'}
  -> tool_call: duckduckgo_search args={'query': 'Brazil orange harvest weather 2025-2026'}
[ToolMessage] Year-over-year growth rates:  [-0.05        0.10

* Можна підібрати кращу модель, можна пропрацювати над інструментами та output форматом.
* Додати джерела (звідки прийшли дані)